# Data Quality Assessment

1. Import Libraries
2. Load Clean Data
3. Dataset Health Scorecard
4. Completeness Analysis
5. Consistency Analysis
6. Validity Analysis
7. Uniqueness Analysis
8. Quality Score
9. Business Risks
10. Final Recommendations

Since this is the last analytical notebook, its purpose is not cleaning (I've already done that in prepare_data()), and it's not EDA (I've already extracted insights).

Its purpose is to answer one question:

"Can management trust this data and what quality issues still exist?"

In [33]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.preprocessing import prepare_data


In [34]:
df = prepare_data()

print(df.shape)

df.head()

(3267, 31)


,Claim ID,Claim Number,Original Amount,Approved Discount %,Recovery Amount,Remaining Amount,Collected Amount,Tp Responsiblity,Debtor Name,Debtor Number,...,Officer,ASCIC Remark,Latest Remark,Unnamed: 24,Unnamed: 25,Recovery Reason,Legal Remark,Location,Debtor Number Status,Recovery Delay (Days)
0,ASCIC03267,C/ERO1/2026/TPSD/0001285,3400.0,NaN,3400.0,3400.0,0.0,0,لجى هادي لحيس العنزي,966506407923,...,MAJED,لا يملك رخصة,NaN,NaN,NaN,لا يملك رخصة قيادة,No legal remarks,Riyadh,Valid,5.0
1,ASCIC03268,C/CRO1/2026/CMC/0010021,6310.2,NaN,6310.2,6310.2,0.0,0.75,منصور سالم سطام الشمري,966506116165,...,MAJED,"استرداد 6,310,20 ريال من المبلغ الزائد الذي تم...",NaN,NaN,NaN,NaN,No legal remarks,Riyadh,Valid,83.0
2,ASCIC03266,C/CRO1/2026/MTP/0007280,1382.0,NaN,1382.0,1382.0,0.0,0,نهاد علي المطر,966502039692,...,MAJED,استرداد مبلغ التعويض الزائد بعد خطأ في التقدير...,NaN,NaN,NaN,NaN,No legal remarks,NaN,Valid,71.0
3,ASCIC03265,C/CRO1/2026/CMC/0005730,9972.0,NaN,9972.0,9972.0,0.0,0,ﻣﻨﻴﺮﻩ ﺣﻤﺪ ﻓﻴﺼﻞ ﺍﻟﻌﺠﻤﻲ,966595266069,...,Munirah,انتهاء رخصة قيادة,NaN,NaN,NaN,أنتهاء تاريخ رخصة القيادة,No legal remarks,Al Ahsa,Valid,130.0
4,ASCIC03248,C/CRO1/2026/PTOA/0013353,4726.8,NaN,4726.8,4726.8,0.0,0,خالد بن فهيد بن عائض السهلي,966503339905,...,Maha Mohammed,عكس السير,NaN,NaN,NaN,عكس اتجاه السير,No legal remarks,Jeddah,Valid,21.0


### Dataset Health Scorecard

In [35]:
total_records = len(df)
total_columns = len(df.columns)

missing_cells = df.isna().sum().sum()

missing_percentage = (
    missing_cells /
    (total_records * total_columns)
) * 100

duplicate_claim_id = df["Claim ID"].duplicated().sum()

duplicate_claim_number = df["Claim Number"].duplicated().sum()

placeholder_numbers = (
    df["Debtor Number Status"] == "Placeholder"
).sum()

unknown_locations = df["Location"].isna().sum()

quality_scorecard = pd.DataFrame({

    "Metric":[
        "Total Records",
        "Total Columns",
        "Missing Cells",
        "Missing %",
        "Duplicate Claim IDs",
        "Duplicate Claim Numbers",
        "Placeholder Debtor Numbers",
        "Unknown Locations"
    ],

    "Value":[
        total_records,
        total_columns,
        missing_cells,
        round(missing_percentage,2),
        duplicate_claim_id,
        duplicate_claim_number,
        placeholder_numbers,
        unknown_locations
    ]

})

quality_scorecard

,Metric,Value
0,Total Records,3267.00
1,Total Columns,31.00
2,Missing Cells,26650.00
3,Missing %,26.31
4,Duplicate Claim IDs,0.00
5,Duplicate Claim Numbers,20.00
6,Placeholder Debtor Numbers,48.00
7,Unknown Locations,20.00


In [36]:
fig = px.bar(

    quality_scorecard,

    x="Metric",

    y="Value",

    title="Dataset Health Scorecard",

    text_auto=".2s"

)

fig.update_layout(

    xaxis_title="",

    yaxis_title=""

)

fig.show()

### Completeness Analysis

In [37]:
completeness = (

    pd.DataFrame({

        "Column":df.columns,

        "Missing":df.isna().sum(),

        "Missing %":(

            df.isna().mean()*100

        ).round(2)

    })

    .sort_values(

        "Missing %",

        ascending=False

    )

)

completeness

,Column,Missing,Missing %
Unnamed: 24,Unnamed: 24,3249,99.45
Unnamed: 25,Unnamed: 25,3245,99.33
Latest Remark,Latest Remark,3245,99.33
Approved Discount %,Approved Discount %,3230,98.87
Reason,Reason,2858,87.48
Driver Iqama,Driver Iqama,2446,74.87
Driver Number,Driver Number,2416,73.95
Driver Name,Driver Name,2327,71.23
ASCIC Remark,ASCIC Remark,665,20.36
Officer,Officer,613,18.76


In [38]:
fig = px.bar(

    completeness.head(15),

    x="Missing %",

    y="Column",

    orientation="h",

    title="Top Columns with Missing Values",

    text="Missing %"

)

fig.show()

### Consistency Analysis

In [39]:
consistency = pd.DataFrame({

    "Check":[

        "Standardized Locations",

        "Standardized Recovery Reasons",

        "Placeholder Debtor Numbers"

    ],

    "Count":[

        df["Location"].notna().sum(),

        df["Recovery Reason"].notna().sum(),

        (
            df["Debtor Number Status"]
            ==
            "Placeholder"
        ).sum()

    ]

})

consistency

,Check,Count
0,Standardized Locations,3247
1,Standardized Recovery Reasons,2956
2,Placeholder Debtor Numbers,48


In [40]:
fig = px.bar(

    consistency,

    x="Check",

    y="Count",

    color="Check",

    title="Consistency Checks"

)

fig.show()

### Validity Analysis

In [41]:
invalid_amounts = (

    df["Collected Amount"]

    >

    df["Recovery Amount"]

).sum()

negative_remaining = (

    df["Remaining Amount"]

    <

    0

).sum()

future_accidents = (

    df["Accident Date"]

    >

    pd.Timestamp.today()

).sum()

validity = pd.DataFrame({

    "Issue":[

        "Collected > Recovery",

        "Negative Remaining",

        "Future Accident Dates"

    ],

    "Count":[

        invalid_amounts,

        negative_remaining,

        future_accidents

    ]

})

validity

,Issue,Count
0,Collected > Recovery,0
1,Negative Remaining,0
2,Future Accident Dates,7


In [42]:
fig = px.bar(

    validity,

    x="Issue",

    y="Count",

    color="Issue",

    title="Validity Checks"

)

fig.show()

### Uniqueness Analysis

In [43]:
uniqueness = pd.DataFrame({

    "Metric":[

        "Duplicate Claim IDs",

        "Duplicate Claim Numbers"

    ],

    "Count":[

        duplicate_claim_id,

        duplicate_claim_number

    ]

})

uniqueness

,Metric,Count
0,Duplicate Claim IDs,0
1,Duplicate Claim Numbers,20


In [44]:
fig = px.pie(

    uniqueness,

    values="Count",

    names="Metric",

    hole=.5,

    title="Duplicate Record Analysis"

)

fig.show()

### Overall Data Quality Score 

In [45]:
quality = 100

quality -= missing_percentage * 0.3

quality -= duplicate_claim_number * 0.2

quality -= placeholder_numbers * 0.05

quality -= unknown_locations * 0.05

quality = max(0, round(quality,2))

print(f"Overall Data Quality Score : {quality}/100")

Overall Data Quality Score : 84.71/100


In [46]:
fig = go.Figure(

    go.Indicator(

        mode="gauge+number",

        value=quality,

        title={"text":"Overall Data Quality Score"},

        gauge={

            "axis":{"range":[0,100]},

            "steps":[

                {"range":[0,50],"color":"red"},

                {"range":[50,75],"color":"orange"},

                {"range":[75,100],"color":"green"}

            ]

        }

    )

)

fig.show()

### Bussiness Risks

In [47]:
business_risks = pd.DataFrame({

    "Risk":[

        "Missing Operational Data",

        "Placeholder Contact Numbers",

        "Duplicate Claim Numbers",

        "Unknown Locations"

    ],

    "Business Impact":[

        "Incomplete Reporting",

        "Customer Follow-up Failure",

        "Investigation Required",

        "Geographic Analysis Impact"

    ]

})

business_risks

,Risk,Business Impact
0,Missing Operational Data,Incomplete Reporting
1,Placeholder Contact Numbers,Customer Follow-up Failure
2,Duplicate Claim Numbers,Investigation Required
3,Unknown Locations,Geographic Analysis Impact


### Final Reccomendations

1. Continue enforcing standardized location mapping.
2. Validate debtor numbers during data entry.
3. Reduce missing operational fields through mandatory forms.
4. Monitor duplicate claim numbers regularly.
5. Automate preprocessing before dashboard refresh.